# pydantic_monty_runtime smoke test

This distribution installs ONE file besides its metadata: the Rust executable `monty`, which `pydantic_monty_client` starts as sandboxed worker processes and which can also be run by hand. It has no Python module to import, so the first cell checks that the `monty` found on PATH is the file this distribution installed and that it starts; the examples then run it. No network, no GPU.

In [ ]:
import importlib.metadata
import os
import shutil
import subprocess

distribution = importlib.metadata.distribution("pydantic_monty_runtime")
version = distribution.version
owned = [path for path in distribution.files if ".dist-info" not in str(path)]
assert len(owned) == 1 and owned[0].name == "monty", owned
installed = os.path.realpath(distribution.locate_file(owned[0]))
assert os.path.isfile(installed) and os.access(installed, os.X_OK), installed
with open(installed, "rb") as handle:
    assert handle.read(4) == b"\x7fELF", "monty is not a compiled program"
on_path = shutil.which("monty")
assert on_path is not None and os.path.realpath(on_path) == installed, (on_path, installed)   # PATH finds THIS file
shown = subprocess.run([installed, "--version"], capture_output=True, text=True, timeout=120)
assert shown.returncode == 0 and shown.stdout.strip() == f"monty-runtime {version}", (shown.returncode, shown.stdout, shown.stderr)
print("pydantic_monty_runtime", version, "installs", installed, "->", shown.stdout.strip())

## Example 1 — run a command and a file

In [ ]:
import os
import subprocess
import tempfile

def monty(*arguments):
    return subprocess.run(["monty", *arguments], capture_output=True, text=True, timeout=120, env={**os.environ, "NO_COLOR": "1"})
one_liner = monty("-c", "print(sum(i * i for i in range(10)))")
assert one_liner.returncode == 0 and one_liner.stdout.splitlines()[0] == "285", (one_liner.returncode, one_liner.stdout, one_liner.stderr)
with tempfile.TemporaryDirectory() as folder:
    script = os.path.join(folder, "double.py")
    with open(script, "w") as handle:
        handle.write("def double(n):\n    return n * 2\nprint(double(21))\n")
    from_file = monty(script)
assert from_file.returncode == 0 and from_file.stdout.splitlines()[0] == "42", (from_file.returncode, from_file.stdout, from_file.stderr)
print("monty -c ->", one_liner.stdout.splitlines()[0], "; monty <file> ->", from_file.stdout.splitlines()[0])

## Example 2 — an error in the code gives a traceback and a non-zero exit status

In [ ]:
import os
import subprocess

failed = subprocess.run(["monty", "-c", "values = [1, 2, 3]\nvalues[10]"], capture_output=True, text=True, timeout=120,
                        env={**os.environ, "NO_COLOR": "1"})
report = failed.stdout + failed.stderr
assert failed.returncode == 1, (failed.returncode, report)
assert "IndexError" in report and "Traceback (most recent call last)" in report, report
print("exit status", failed.returncode, "with:", [line for line in report.splitlines() if "IndexError" in line][0].strip())

## Example 3 — the type checker reports a wrong assignment when asked

In [ ]:
import os
import subprocess

def monty(*arguments):
    return subprocess.run(["monty", *arguments], capture_output=True, text=True, timeout=120, env={**os.environ, "NO_COLOR": "1"})
wrong = monty("-t", "-c", 'count: int = "three"')
report = wrong.stdout + wrong.stderr
# The command-line program REPORTS a type error and still runs the code (exit status 0); the Python API refuses instead.
assert "type check failed" in report and "invalid-assignment" in report, (wrong.returncode, report)
right = monty("-t", "-c", "count: int = 3\nprint(count + 1)")
report = right.stdout + right.stderr
assert right.returncode == 0 and right.stdout.splitlines()[0] == "4" and "type check failed" not in report, (right.returncode, report)
print("type error reported for the wrong program; the correct one passed the check and printed", right.stdout.splitlines()[0])

## Example 4 — the sandbox walls: host files and the environment are out of reach unless a folder is mounted

In [ ]:
import os
import subprocess
import tempfile

def monty(*arguments):
    return subprocess.run(["monty", *arguments], capture_output=True, text=True, timeout=120, env={**os.environ, "NO_COLOR": "1"})
for code in ('print(open("/etc/hostname").read())', 'import os\nprint(os.environ)'):
    walled = monty("-c", code)
    assert walled.returncode == 1 and "not implemented" in (walled.stdout + walled.stderr), (code, walled.stdout, walled.stderr)
with tempfile.TemporaryDirectory() as folder:
    with open(os.path.join(folder, "note.txt"), "w") as handle:
        handle.write("mounted read-only")
    mounted = monty("--mount", f"{folder}::/data::ro", "-c", 'print(open("/data/note.txt").read())')
    assert mounted.returncode == 0 and mounted.stdout.splitlines()[0] == "mounted read-only", (mounted.stdout, mounted.stderr)
    blocked = monty("--mount", f"{folder}::/data::ro", "-c", 'open("/data/new.txt", "w").write("x")')
    assert blocked.returncode == 1 and not os.path.exists(os.path.join(folder, "new.txt")), (blocked.stdout, blocked.stderr)
print("no host file or environment without a mount; a read-only mount can be read and not written")

## Example 5 — limits stop runaway code

In [ ]:
import os
import subprocess

def monty(*arguments):
    return subprocess.run(["monty", *arguments], capture_output=True, text=True, timeout=120, env={**os.environ, "NO_COLOR": "1"})
stopped = {}
for label, options, code, expected in (
        ("endless loop", ["--max-feed-duration", "0.2"], "while True:\n    pass", "TimeoutError"),
        ("memory", ["--max-memory", "1MB"], "blob = [0] * 50_000_000", "MemoryError"),
        ("recursion", ["--max-recursion-depth", "50"], "def down(n):\n    return down(n + 1)\ndown(0)", "RecursionError")):
    result = monty(*options, "-c", code)
    report = result.stdout + result.stderr
    assert result.returncode == 1 and expected in report, (label, result.returncode, report[-600:])
    stopped[label] = expected
print("stopped:", stopped)